In [5]:
class AccountFrozenError(Exception):
    """Операция невозможна: счёт заморожен"""
    pass


class AccountClosedError(Exception):
    """Операция невозможна: счёт закрыт"""
    pass


class InvalidOperationError(Exception):
    """Некорректные входные данные операции"""
    pass


class InsufficientFundsError(Exception):
    """Недостаточно средств на счёте"""
    pass

In [6]:
from abc import ABC, abstractmethod


class AbstractAccount(ABC):
    """
    Абстрактная модель банковского счёта.
    Задаёт контракт, который обязаны реализовать все конкретные типы счетов.
    """

    def __init__(self, account_id: str, owner: str):
        self._account_id = account_id
        self._owner = owner
        self._balance = 0.0
        self._status = "active"  # active | frozen | closed

    @abstractmethod
    def deposit(self, amount: float):
        """Пополнение счёта"""
        pass

    @abstractmethod
    def withdraw(self, amount: float):
        """Снятие средств со счёта"""
        pass

    @abstractmethod
    def get_account_info(self) -> dict:
        """Возвращает информацию о счёте"""
        pass

In [7]:
import uuid


VALID_CURRENCIES = {"RUB", "USD", "EUR", "KZT", "CNY"}


def generate_short_uuid() -> str:
    """Генерирует короткий уникальный идентификатор счёта"""
    return str(uuid.uuid4())[:8]

In [8]:
class BankAccount(AbstractAccount):
    """Конкретная реализация банковского счёта"""

    def __init__(self, owner: str, currency: str = "RUB", account_id: str = None):
        # если номер счёта не передали - генерируем сами
        if not account_id:
            account_id = generate_short_uuid()

        # валидация валюты если что-то не так с валютой то выпадает ошибка Неподдерживаемая валюта и показываются Допустимые валюты
        if currency not in VALID_CURRENCIES:
            raise InvalidOperationError(
                f"Неподдерживаемая валюта: {currency}. "
                f"Допустимые: {', '.join(VALID_CURRENCIES)}"
            )

        # not owner сработает (будет True), если owner — это пустая строка "" или None + тут еще проверка типа данных имени - строка ли
        if not owner or not isinstance(owner, str):
            raise InvalidOperationError("Имя владельца должно быть непустой строкой")

        super().__init__(account_id, owner)
        self._currency = currency

    def _check_active(self):
        """Проверка, что со счётом можно работать"""
        if self._status == "frozen":
            raise AccountFrozenError(f"Счёт {self._account_id} заморожен")
        if self._status == "closed":
            raise AccountClosedError(f"Счёт {self._account_id} закрыт")

    def deposit(self, amount: float):
        self._check_active()

        if not isinstance(amount, (int, float)) or amount <= 0:
            raise InvalidOperationError("Сумма пополнения должна быть положительным числом")

        self._balance += amount
        return self._balance

    def withdraw(self, amount: float):
        self._check_active()

        if not isinstance(amount, (int, float)) or amount <= 0:
            raise InvalidOperationError("Сумма снятия должна быть положительным числом")

        if amount > self._balance:
            raise InsufficientFundsError(
                f"Недостаточно средств. Баланс: {self._balance}, запрошено: {amount}"
            )

        self._balance -= amount
        return self._balance

    def get_account_info(self) -> dict:
        return {
            "account_id": self._account_id,
            "owner": self._owner,
            "balance": self._balance,
            "currency": self._currency,
            "status": self._status,
        }

    def freeze(self):
        """Заморозить счёт"""
        self._status = "frozen"

    def close(self):
        """Закрыть счёт"""
        self._status = "closed"

    def __str__(self):
        last_four = self._account_id[-4:]
        return (
            f"BankAccount(владелец={self._owner}, "
            f"№***{last_four}, статус={self._status}, "
            f"баланс={self._balance} {self._currency})"
        )

In [11]:
print("=== Демонстрация работы BankAccount ===\n")

# 1. Создание активного счёта
print("1. Создаём активный счёт")
active_account = BankAccount(owner="Екатерина Кириллина", currency="RUB")
print(active_account)
print()

# 2. Создание счёта и заморозка
print("2. Создаём счёт и замораживаем его")
frozen_account = BankAccount(owner="Иван Иванов", currency="USD")
frozen_account.freeze()
print(frozen_account)
print()

# 3. Попытка операций над замороженным счётом
print("3. Пытаемся пополнить замороженный счёт")
try:
    frozen_account.deposit(100)
except AccountFrozenError as e:
    print(f"Ошибка (ожидаемо): {e}")
print()

print("4. Пытаемся снять деньги с замороженного счёта")
try:
    frozen_account.withdraw(50)
except AccountFrozenError as e:
    print(f"Ошибка (ожидаемо): {e}")
print()

# 5. Валидное пополнение и снятие на активном счёте
print("5. Валидное пополнение активного счёта")
active_account.deposit(1000)
print(active_account)
print()

print("6. Валидное снятие с активного счёта")
active_account.withdraw(300)
print(active_account)
print()

# 7. Попытка снять больше, чем есть на балансе
print("7. Пытаемся снять больше, чем есть на счёте")
try:
    active_account.withdraw(10000)
except InsufficientFundsError as e:
    print(f"Ошибка (ожидаемо): {e}")
print()

# 8. Получение информации о счёте
print("8. Информация о счёте (get_account_info)")
print(active_account.get_account_info())

=== Демонстрация работы BankAccount ===

1. Создаём активный счёт
BankAccount(владелец=Екатерина Кириллина, №***c4f8, статус=active, баланс=0.0 RUB)

2. Создаём счёт и замораживаем его
BankAccount(владелец=Иван Иванов, №***21a1, статус=frozen, баланс=0.0 USD)

3. Пытаемся пополнить замороженный счёт
Ошибка (ожидаемо): Счёт 334621a1 заморожен

4. Пытаемся снять деньги с замороженного счёта
Ошибка (ожидаемо): Счёт 334621a1 заморожен

5. Валидное пополнение активного счёта
BankAccount(владелец=Екатерина Кириллина, №***c4f8, статус=active, баланс=1000.0 RUB)

6. Валидное снятие с активного счёта
BankAccount(владелец=Екатерина Кириллина, №***c4f8, статус=active, баланс=700.0 RUB)

7. Пытаемся снять больше, чем есть на счёте
Ошибка (ожидаемо): Недостаточно средств. Баланс: 700.0, запрошено: 10000

8. Информация о счёте (get_account_info)
{'account_id': 'b58cc4f8', 'owner': 'Екатерина Кириллина', 'balance': 700.0, 'currency': 'RUB', 'status': 'active'}
